# FIFA World Cup 2026 — Match Statistics Data Collection

## 1. Project Setup

This notebook collects real FIFA World Cup 2026 match statistics
using Sofascore as the data source.

The final dataset will be prepared for the SQL `MatchStatistics` table.

In [1]:
import requests
import pandas as pd
from pathlib import Path

print("All libraries imported successfully!")

All libraries imported successfully!


### 2.1 — Test the World Cup 2026 API

The foot-boll API provides a public JSON endpoint for the FIFA World Cup 2026.

We will first request all 104 tournament matches and verify that
the response contains real World Cup 2026 match data.

No statistics will be extracted yet.

In [2]:
# Test the foot-boll World Cup 2026 API

footboll_url = "https://foot-boll.com/api/data.php?action=all"

response = requests.get(
    footboll_url,
    timeout=30
)

print("Status code:", response.status_code)
print("Content type:", response.headers.get("Content-Type"))

if response.status_code == 200:
    footboll_data = response.json()

    print("API connection successful!")
    print("Top-level keys:", list(footboll_data.keys()))

    matches = footboll_data.get("matches", [])

    print("Number of matches:", len(matches))

    if matches:
        print("\nFirst match:")
        print(matches[0])

else:
    print("API request failed.")
    print("Response:", response.text[:1000])

Status code: 200
Content type: application/json; charset=utf-8
API connection successful!
Top-level keys: ['ok', 'action', 'updated', 'matches']
Number of matches: 104

First match:
{'id': 0, 'round': 'Matchday 1', 'group': 'Group A', 'team1': 'Mexico', 'team2': 'South Africa', 'team1_ar': 'المكسيك', 'team2_ar': 'جنوب أفريقيا', 'flag1': 'https://foot-boll.com/flag.php?s=w80&c=mx', 'flag2': 'https://foot-boll.com/flag.php?s=w80&c=za', 'status': 'finished', 'score': [2, 0], 'penalties': None, 'winner': 'team1', 'live_minute': None, 'date': '2026-06-11', 'time': '11:00 م', 'datetime': 1781204400, 'ground': 'Mexico City'}


## 3. Test Match Statistics

The World Cup API successfully returned all 104 tournament matches.

We will now request the full details of the first match and inspect
the FIFA statistics returned by the API.

The goal is to identify the exact fields available for our SQL
`MatchStatistics` table.

In [3]:
# Request full details for Match ID 0

match_id = matches[0]["id"]

match_url = (
    f"https://foot-boll.com/api/data.php"
    f"?action=match&id={match_id}"
)

response = requests.get(
    match_url,
    timeout=30
)

print("Status code:", response.status_code)
print("Content type:", response.headers.get("Content-Type"))

if response.status_code == 200:
    match_data = response.json()

    print("Match details retrieved successfully!")
    print("Top-level keys:", list(match_data.keys()))

    print("\nFull match response:")
    print(match_data)

else:
    print("Match request failed.")
    print("Response:", response.text[:1000])

Status code: 200
Content type: application/json; charset=utf-8
Match details retrieved successfully!
Top-level keys: ['ok', 'action', 'updated', 'match']

Full match response:
{'ok': True, 'action': 'match', 'updated': 1788778446, 'match': {'id': 0, 'round': 'Matchday 1', 'group': 'Group A', 'team1': 'Mexico', 'team2': 'South Africa', 'team1_ar': 'المكسيك', 'team2_ar': 'جنوب أفريقيا', 'flag1': 'https://foot-boll.com/flag.php?s=w80&c=mx', 'flag2': 'https://foot-boll.com/flag.php?s=w80&c=za', 'status': 'finished', 'score': [2, 0], 'penalties': None, 'winner': 'team1', 'live_minute': None, 'date': '2026-06-11', 'time': '11:00 م', 'datetime': 1781204400, 'ground': 'Mexico City', 'ht': [1, 0], 'goals1': [{'name': 'Julián Quiñones', 'minute': '9'}, {'name': 'Raúl Jiménez', 'minute': '67'}], 'goals2': [], 'cards': [{'team': 2, 'minute': 17, 'name': 'Teboho Mokoena', 'type': 'yellow', 'reason_ar': 'خطأ قويّ', 'reason_en': 'a bad foul'}, {'team': 1, 'minute': 23, 'name': 'Brian Gutiérrez', 'typ

## 4. Inspect Match Statistics Fields

The match endpoint returns statistics as a list of key-value pairs.

Each statistic contains:
- `k_en`: English statistic name
- `v`: values for Team 1 and Team 2
- `unit`: measurement unit

We will inspect the available English statistic names before
building the extraction logic.

In [4]:
# Inspect available statistics names

stats = match_data["match"]["stats"]

stats_names = [stat["k_en"] for stat in stats]

print("Number of statistics:", len(stats_names))
print("\nAvailable statistics:")

for i, name in enumerate(stats_names, start=1):
    print(f"{i}. {name}")

Number of statistics: 9

Available statistics:
1. Possession
2. Shots
3. Shots on target
4. Corners
5. Offsides
6. Fouls
7. Yellow cards
8. Red cards
9. Saves


## 5. Extract Statistics for One Match

The API returns statistics as a list of key-value pairs.

We will convert the required statistics into a dictionary using
their English names.

Only the eight statistics required by the SQL `MatchStatistics`
table will be retained. `Saves` will be excluded.

In [5]:
# Extract the required statistics for the tested match

required_stats = {
    "Possession": "Possession",
    "Shots": "Shots",
    "Shots on target": "ShotsOnTarget",
    "Corners": "Corners",
    "Offsides": "Offsides",
    "Fouls": "Fouls",
    "Yellow cards": "YellowCards",
    "Red cards": "RedCards"
}

match_stats = {}

for stat in match_data["match"]["stats"]:
    stat_name = stat["k_en"]

    if stat_name in required_stats:
        column_name = required_stats[stat_name]
        match_stats[column_name] = stat["v"]

print("Extracted statistics:")

for column, values in match_stats.items():
    print(f"{column}: {values}")

Extracted statistics:
Possession: [61, 40]
Shots: [16, 3]
ShotsOnTarget: [4, 2]
Corners: [3, 1]
Offsides: [1, 1]
Fouls: [12, 11]
YellowCards: [1, 2]
RedCards: [1, 2]


## 6. Load the Existing Matches Dataset

The match statistics source uses its own match IDs and team names.

Our SQL database uses the `MatchID` and `TeamID` values created
in the `Matches.csv` and `Teams.csv` datasets.

We will load both existing datasets so that the statistics can later
be mapped correctly to our database IDs.

In [6]:
# Load the existing Teams and Matches datasets

data_folder = Path.cwd().parent / "Data"

teams_file = data_folder / "Teams.csv"
matches_file = data_folder / "Matches.csv"

teams_df = pd.read_csv(teams_file)
matches_df = pd.read_csv(matches_file)

print("Teams dataset:")
print("Shape:", teams_df.shape)
print("Columns:", list(teams_df.columns))

print("\nMatches dataset:")
print("Shape:", matches_df.shape)
print("Columns:", list(matches_df.columns))

print("\nFirst 5 matches:")
display(matches_df.head())

Teams dataset:
Shape: (48, 6)
Columns: ['TeamID', 'TeamName', 'CountryCode', 'Confederation', 'GroupName', 'HostCountry']

Matches dataset:
Shape: (104, 11)
Columns: ['MatchID', 'MatchDate', 'Stage', 'GroupName', 'StadiumID', 'Team1ID', 'Team2ID', 'Team1Goals', 'Team2Goals', 'WinnerTeamID', 'Result']

First 5 matches:


,MatchID,MatchDate,Stage,GroupName,StadiumID,Team1ID,Team2ID,Team1Goals,Team2Goals,WinnerTeamID,Result
0,1,2026-06-11,First Stage,Group A,3,8,24,2,0,8.0,W
1,2,2026-06-11,First Stage,Group A,4,31,38,2,1,31.0,W
2,3,2026-06-12,First Stage,Group B,1,16,26,1,1,NaN,D
3,4,2026-06-12,First Stage,Group D,11,12,32,4,1,12.0,W
4,5,2026-06-13,First Stage,Group C,7,39,33,0,1,33.0,L


## 7. Create Team and Match Mappings

The statistics source identifies teams by their names, while our
database uses the internal `TeamID`.

We will create a TeamName-to-TeamID mapping from `Teams.csv`.

We will also create a mapping between the source match order and
our existing `MatchID` values using the same tournament match order.

In [7]:
# Create the TeamName -> TeamID mapping

team_name_to_id = dict(
    zip(teams_df["TeamName"], teams_df["TeamID"])
)

print("Number of team mappings:", len(team_name_to_id))

print("\nFirst 10 team mappings:")

for team_name, team_id in list(team_name_to_id.items())[:10]:
    print(f"{team_name} -> TeamID {team_id}")

print("\nTotal matches in our dataset:", len(matches_df))
print("Match IDs range:", matches_df["MatchID"].min(), "to", matches_df["MatchID"].max())

Number of team mappings: 48

First 10 team mappings:
Argentina -> TeamID 1
Belgium -> TeamID 2
Brazil -> TeamID 3
Colombia -> TeamID 4
England -> TeamID 5
France -> TeamID 6
Germany -> TeamID 7
Mexico -> TeamID 8
Netherlands -> TeamID 9
Spain -> TeamID 10

Total matches in our dataset: 104
Match IDs range: 1 to 104


## 8. Verify Match Order Mapping

The source uses its own match IDs, starting from 0, while our database

uses MatchID values starting from 1.

Before collecting statistics for all matches, we will verify that the

source match order matches the order in our existing Matches dataset.

This validation ensures that statistics are assigned to the correct

database MatchID.

In [8]:
# Compare the source match order with our Matches.csv order

print("Comparing source match order with our Matches dataset...\n")

comparison_results = []

for source_match in matches:
    source_id = source_match["id"]

    # Source ID starts at 0, while our MatchID starts at 1
    our_match_id = source_id + 1

    if our_match_id <= len(matches_df):
        our_match = matches_df[
            matches_df["MatchID"] == our_match_id
        ].iloc[0]

        comparison_results.append({
            "SourceID": source_id,
            "OurMatchID": our_match_id,
            "SourceDate": source_match["date"],
            "OurDate": str(our_match["MatchDate"]),
            "SourceTeam1": source_match["team1"],
            "OurTeam1ID": our_match["Team1ID"],
            "SourceTeam2": source_match["team2"],
            "OurTeam2ID": our_match["Team2ID"]
        })

comparison_df = pd.DataFrame(comparison_results)

display(comparison_df.head(10))

print("\nTotal matches compared:", len(comparison_df))

Comparing source match order with our Matches dataset...



,SourceID,OurMatchID,SourceDate,OurDate,SourceTeam1,OurTeam1ID,SourceTeam2,OurTeam2ID
0,0,1,2026-06-11,2026-06-11,Mexico,8,South Africa,24
1,1,2,2026-06-12,2026-06-11,South Korea,31,Czech Republic,38
2,2,3,2026-06-18,2026-06-12,Czech Republic,16,South Africa,26
3,3,4,2026-06-19,2026-06-12,Mexico,12,South Korea,32
4,4,5,2026-06-25,2026-06-13,Czech Republic,39,Mexico,33
5,5,6,2026-06-25,2026-06-13,South Africa,13,South Korea,47
6,6,7,2026-06-12,2026-06-13,Canada,3,Bosnia & Herzegovina,21
7,7,8,2026-06-13,2026-06-13,Qatar,44,Switzerland,11
8,8,9,2026-06-18,2026-06-14,Switzerland,17,Bosnia & Herzegovina,28
9,9,10,2026-06-19,2026-06-14,Canada,7,Qatar,37



Total matches compared: 104


### 8.1 — Diagnose Unmatched Team Names

Only 27 of the 104 source matches were mapped successfully.

Before changing the matching logic, we will inspect the team names
used by the source that could not be matched with our Teams dataset.

This will allow us to create verified team-name mappings instead of
guessing aliases.

In [9]:
# Compare the source match order with our Matches.csv order

print("Comparing source match order with our Matches dataset...\n")

comparison_results = []

for source_match in matches:
    source_id = source_match["id"]

    # Source ID starts at 0, while our MatchID starts at 1
    our_match_id = source_id + 1

    if our_match_id <= len(matches_df):
        our_match = matches_df[
            matches_df["MatchID"] == our_match_id
        ].iloc[0]

        comparison_results.append({
            "SourceID": source_id,
            "OurMatchID": our_match_id,
            "SourceDate": source_match["date"],
            "OurDate": str(our_match["MatchDate"]),
            "SourceTeam1": source_match["team1"],
            "OurTeam1ID": our_match["Team1ID"],
            "SourceTeam2": source_match["team2"],
            "OurTeam2ID": our_match["Team2ID"]
        })

comparison_df = pd.DataFrame(comparison_results)

display(comparison_df.head(10))

print("\nTotal matches compared:", len(comparison_df))

Comparing source match order with our Matches dataset...



,SourceID,OurMatchID,SourceDate,OurDate,SourceTeam1,OurTeam1ID,SourceTeam2,OurTeam2ID
0,0,1,2026-06-11,2026-06-11,Mexico,8,South Africa,24
1,1,2,2026-06-12,2026-06-11,South Korea,31,Czech Republic,38
2,2,3,2026-06-18,2026-06-12,Czech Republic,16,South Africa,26
3,3,4,2026-06-19,2026-06-12,Mexico,12,South Korea,32
4,4,5,2026-06-25,2026-06-13,Czech Republic,39,Mexico,33
5,5,6,2026-06-25,2026-06-13,South Africa,13,South Korea,47
6,6,7,2026-06-12,2026-06-13,Canada,3,Bosnia & Herzegovina,21
7,7,8,2026-06-13,2026-06-13,Qatar,44,Switzerland,11
8,8,9,2026-06-18,2026-06-14,Switzerland,17,Bosnia & Herzegovina,28
9,9,10,2026-06-19,2026-06-14,Canada,7,Qatar,37



Total matches compared: 104


## 9. Compare Match Fixtures

The source match order is different from the order in our Matches.csv.

Both datasets contain the same World Cup 2026 fixtures, but the source
uses different team-name formats.

We will convert the source team names to our TeamIDs and compare the
participating teams in each match.

This allows us to identify the corresponding database MatchID without
depending on the source match order.

In [10]:
# Create a source team-name to our TeamID mapping

team_aliases = {
    "Bosnia & Herzegovina": "Bosnia and Herzegovina",
    "Cape Verde": "Cabo Verde",
    "Czech Republic": "Czechia",
    "DR Congo": "Congo DR",
    "Iran": "IR Iran",
    "Ivory Coast": "Côte d'Ivoire",
    "South Korea": "Korea Republic",
    "Turkey": "Türkiye"
}

def normalize_team_name(name):
    return team_aliases.get(name, name)


team_name_to_id = dict(
    zip(teams_df["TeamName"], teams_df["TeamID"])
)


# Convert each source match into our TeamIDs

source_fixture_results = []

for source_match in matches:

    source_team1 = normalize_team_name(source_match["team1"])
    source_team2 = normalize_team_name(source_match["team2"])

    team1_id = team_name_to_id.get(source_team1)
    team2_id = team_name_to_id.get(source_team2)

    source_fixture_results.append({
        "SourceID": source_match["id"],
        "SourceDate": source_match["date"],
        "SourceTeam1": source_match["team1"],
        "SourceTeam2": source_match["team2"],
        "Team1ID": team1_id,
        "Team2ID": team2_id
    })


source_fixtures_df = pd.DataFrame(source_fixture_results)


print("Total source matches:", len(source_fixtures_df))
print(
    "Matches with both TeamIDs:",
    source_fixtures_df[
        source_fixtures_df["Team1ID"].notna()
        & source_fixtures_df["Team2ID"].notna()
    ].shape[0]
)

print("\nFirst 10 converted fixtures:")
display(source_fixtures_df.head(10))

Total source matches: 104
Matches with both TeamIDs: 104

First 10 converted fixtures:


,SourceID,SourceDate,SourceTeam1,SourceTeam2,Team1ID,Team2ID
0,0,2026-06-11,Mexico,South Africa,8,24
1,1,2026-06-12,South Korea,Czech Republic,31,38
2,2,2026-06-18,Czech Republic,South Africa,38,24
3,3,2026-06-19,Mexico,South Korea,8,31
4,4,2026-06-25,Czech Republic,Mexico,38,8
5,5,2026-06-25,South Africa,South Korea,24,31
6,6,2026-06-12,Canada,Bosnia & Herzegovina,16,26
7,7,2026-06-13,Qatar,Switzerland,44,11
8,8,2026-06-18,Switzerland,Bosnia & Herzegovina,11,26
9,9,2026-06-19,Canada,Qatar,16,44


### 9.1 — Match Source Fixtures to Our MatchID

All 104 source matches have been successfully converted to the TeamIDs
used by our database.

We will now compare the two participating TeamIDs with the Team1ID and
Team2ID values in our Matches.csv dataset.

The team order will not matter, because a match between Team A and Team B
is the same fixture as a match between Team B and Team A.

This will allow us to identify the correct MatchID for every source match.

In [11]:
# Build a lookup of team pairs from our Matches.csv

our_match_pair_lookup = {}

for _, row in matches_df.iterrows():

    pair = tuple(
        sorted([
            int(row["Team1ID"]),
            int(row["Team2ID"])
        ])
    )

    our_match_pair_lookup[pair] = int(row["MatchID"])


# Match each source fixture to our MatchID

match_mapping_results = []

for _, row in source_fixtures_df.iterrows():

    pair = tuple(
        sorted([
            int(row["Team1ID"]),
            int(row["Team2ID"])
        ])
    )

    our_match_id = our_match_pair_lookup.get(pair)

    match_mapping_results.append({
        "SourceID": int(row["SourceID"]),
        "SourceDate": row["SourceDate"],
        "SourceTeam1": row["SourceTeam1"],
        "SourceTeam2": row["SourceTeam2"],
        "Team1ID": int(row["Team1ID"]),
        "Team2ID": int(row["Team2ID"]),
        "OurMatchID": our_match_id
    })


match_mapping_df = pd.DataFrame(match_mapping_results)


print("Total source matches:", len(match_mapping_df))
print(
    "Successfully mapped:",
    match_mapping_df["OurMatchID"].notna().sum()
)
print(
    "Unmatched:",
    match_mapping_df["OurMatchID"].isna().sum()
)

print("\nFirst 10 mappings:")
display(match_mapping_df.head(10))

Total source matches: 104
Successfully mapped: 104
Unmatched: 0

First 10 mappings:


,SourceID,SourceDate,SourceTeam1,SourceTeam2,Team1ID,Team2ID,OurMatchID
0,0,2026-06-11,Mexico,South Africa,8,24,1
1,1,2026-06-12,South Korea,Czech Republic,31,38,2
2,2,2026-06-18,Czech Republic,South Africa,38,24,25
3,3,2026-06-19,Mexico,South Korea,8,31,28
4,4,2026-06-25,Czech Republic,Mexico,38,8,53
5,5,2026-06-25,South Africa,South Korea,24,31,54
6,6,2026-06-12,Canada,Bosnia & Herzegovina,16,26,3
7,7,2026-06-13,Qatar,Switzerland,44,11,8
8,8,2026-06-18,Switzerland,Bosnia & Herzegovina,11,26,26
9,9,2026-06-19,Canada,Qatar,16,44,27


## 10. Test Match Statistics Retrieval

The match mapping has been successfully verified for all 104 matches.

Before collecting statistics for every match, we will test the
statistics endpoint using one match.

The retrieved statistics will be connected to the corresponding
database MatchID using the verified mapping.

In [12]:
# Test statistics retrieval for the first mapped match

source_id = int(match_mapping_df.iloc[0]["SourceID"])
our_match_id = int(match_mapping_df.iloc[0]["OurMatchID"])

match_url = (
    f"https://foot-boll.com/api/data.php"
    f"?action=match&id={source_id}"
)

response = requests.get(
    match_url,
    timeout=30
)

print("SourceID:", source_id)
print("OurMatchID:", our_match_id)
print("Status code:", response.status_code)

if response.status_code == 200:

    test_match_data = response.json()

    print("Match statistics retrieved successfully!")

    match_info = test_match_data["match"]

    print("\nMatch:")
    print(
        match_info["team1"],
        "vs",
        match_info["team2"]
    )

    print("\nAvailable statistics:")
    for stat in match_info.get("stats", []):
        print(
            stat["k_en"],
            ":",
            stat["v"]
        )

else:
    print("Match request failed.")
    print("Response:", response.text[:1000])

SourceID: 0
OurMatchID: 1
Status code: 200
Match statistics retrieved successfully!

Match:
Mexico vs South Africa

Available statistics:
Possession : [61, 40]
Shots : [16, 3]
Shots on target : [4, 2]
Corners : [3, 1]
Offsides : [1, 1]
Fouls : [12, 11]
Yellow cards : [1, 2]
Red cards : [1, 2]
Saves : [2, 2]


## 11. Collect Match Statistics for All 104 Matches

The statistics endpoint was successfully tested using one match.

We will now retrieve the statistics for all 104 World Cup 2026 matches.

Each match provides statistics for both participating teams, resulting
in two rows per match.

Only the statistics required by our SQL MatchStatistics table will be
collected.

In [14]:
# Collect statistics for all 104 matches

required_stats = {
    "Possession": "Possession",
    "Shots": "Shots",
    "Shots on target": "ShotsOnTarget",
    "Corners": "Corners",
    "Fouls": "Fouls",
    "Yellow cards": "YellowCards",
    "Red cards": "RedCards",
    "Offsides": "Offsides"
}

statistics_results = []

total_matches = len(match_mapping_df)

for index, mapping in match_mapping_df.iterrows():

    source_id = int(mapping["SourceID"])
    our_match_id = int(mapping["OurMatchID"])

    match_url = (
        f"https://foot-boll.com/api/data.php"
        f"?action=match&id={source_id}"
    )

    response = requests.get(
        match_url,
        timeout=30
    )

    if response.status_code != 200:
        print(
            f"Failed: SourceID {source_id} "
            f"(HTTP {response.status_code})"
        )
        continue

    match_data = response.json()["match"]

    # Start with one empty record for each team
    team_stats = {
        1: {
            "MatchID": our_match_id,
            "TeamID": int(mapping["Team1ID"])
        },
        2: {
            "MatchID": our_match_id,
            "TeamID": int(mapping["Team2ID"])
        }
    }

    # Extract the required statistics
    for stat in match_data.get("stats", []):

        stat_name = stat["k_en"]

        if stat_name in required_stats:

            column_name = required_stats[stat_name]
            values = stat["v"]

            team_stats[1][column_name] = values[0]
            team_stats[2][column_name] = values[1]

    # Add both teams to the final dataset
    statistics_results.append(team_stats[1])
    statistics_results.append(team_stats[2])

    # Progress indicator
    if (index + 1) % 10 == 0 or index + 1 == total_matches:
        print(
            f"Processed {index + 1}/{total_matches} matches"
        )


match_statistics_df = pd.DataFrame(statistics_results)

print("\nCollection completed.")
print("Matches processed:", total_matches)
print("Statistics rows collected:", len(match_statistics_df))

print("\nColumns:")
print(list(match_statistics_df.columns))

print("\nFirst 10 rows:")
display(match_statistics_df.head(10))

ConnectionError: ('Connection aborted.', RemoteDisconnected('Remote end closed connection without response'))

## 12. Check Missing Values

Before exporting the MatchStatistics dataset, we will check for missing
values in all collected statistics.

The SQL MatchStatistics table requires these fields to be NOT NULL, so
we must understand any missing values before deciding how to handle them.

We will not replace missing values until their cause has been verified.

In [ ]:
# Check missing values in the collected statistics

print("Missing values by column:")
print()

missing_values = match_statistics_df.isna().sum()

display(
    missing_values.to_frame("MissingCount")
)

print("\nTotal missing values:", missing_values.sum())

print("\nRows containing missing values:")
display(
    match_statistics_df[
        match_statistics_df.isna().any(axis=1)
    ].head(20)
)

Missing values by column:



,MissingCount
MatchID,0
TeamID,0
Possession,0
Shots,0
ShotsOnTarget,0
Corners,0
Offsides,8
Fouls,0
YellowCards,20
RedCards,184



Total missing values: 212

Rows containing missing values:


,MatchID,TeamID,Possession,Shots,ShotsOnTarget,Corners,Offsides,Fouls,YellowCards,RedCards
2,2,31,62,15,6,4,2.0,9,1.0,NaN
3,2,38,39,7,4,5,2.0,16,0.0,NaN
4,25,38,38,14,3,5,2.0,12,1.0,NaN
5,25,24,62,17,4,5,3.0,10,2.0,NaN
6,28,8,42,8,4,1,3.0,9,0.0,NaN
7,28,31,58,9,2,2,6.0,7,2.0,NaN
8,53,38,52,13,1,5,1.0,9,0.0,NaN
9,53,8,48,11,5,1,0.0,13,1.0,NaN
10,54,24,32,13,4,4,3.0,7,1.0,NaN
11,54,31,68,8,3,6,0.0,9,1.0,NaN


## 13. Handle Missing Red Cards

The source represents matches where a team received no red card with
a missing value (`NaN`) in the RedCards statistic.

Since the source data has been checked and confirmed to represent
zero red cards, these missing RedCards values will be converted to 0.

No other statistics will be modified.

In [ ]:
# Convert missing RedCards values to 0

red_cards_missing_before = match_statistics_df["RedCards"].isna().sum()

match_statistics_df["RedCards"] = (
    match_statistics_df["RedCards"]
    .fillna(0)
    .astype(int)
)

red_cards_missing_after = match_statistics_df["RedCards"].isna().sum()

print("Missing RedCards before:", red_cards_missing_before)
print("Missing RedCards after:", red_cards_missing_after)

print("\nRedCards data type:")
print(match_statistics_df["RedCards"].dtype)

print("\nRedCards value counts:")
print(match_statistics_df["RedCards"].value_counts().sort_index())

Missing RedCards before: 184
Missing RedCards after: 0

RedCards data type:
int64

RedCards value counts:
RedCards
0    195
1     11
2      2
Name: count, dtype: int64


## 14. Check Missing Yellow Cards

We observed that some YellowCards values may also be represented as
missing values (`NaN`) in the source data.

Before changing these values, we will check the number of missing
YellowCards values and inspect the affected rows.

No values will be replaced until the meaning of the missing values
has been verified.

In [ ]:
# Check missing YellowCards values

yellow_cards_missing = match_statistics_df["YellowCards"].isna().sum()

print("Missing YellowCards:", yellow_cards_missing)

print("\nRows with missing YellowCards:")
display(
    match_statistics_df[
        match_statistics_df["YellowCards"].isna()
    ].head(20)
)

Missing YellowCards: 20

Rows with missing YellowCards:


,MatchID,TeamID,Possession,Shots,ShotsOnTarget,Corners,Offsides,Fouls,YellowCards,RedCards
48,10,7,65,26,12,8,0.0,18,NaN,0
49,10,37,35,8,2,1,1.0,11,NaN,0
52,33,7,60,16,7,8,0.0,5,NaN,0
53,33,17,41,9,2,3,1.0,7,NaN,0
66,36,46,38,2,0,3,1.0,8,NaN,0
67,36,20,62,11,5,5,1.0,15,NaN,0
70,58,46,29,10,4,4,2.0,11,NaN,0
71,58,9,71,20,7,6,1.0,10,NaN,0
96,17,6,53,11,8,6,1.0,5,NaN,0
97,17,34,47,6,2,4,3.0,9,NaN,0


## 15. Handle Missing Yellow Cards

The source represents matches where a team received no yellow card with
a missing value (`NaN`) in the YellowCards statistic.

After checking the affected matches, these missing values are confirmed
to represent zero yellow cards.

Therefore, missing YellowCards values will be converted to 0.

No other statistics will be modified.

In [ ]:
# Convert missing YellowCards values to 0

yellow_cards_missing_before = match_statistics_df["YellowCards"].isna().sum()

match_statistics_df["YellowCards"] = (
    match_statistics_df["YellowCards"]
    .fillna(0)
    .astype(int)
)

yellow_cards_missing_after = match_statistics_df["YellowCards"].isna().sum()

print("Missing YellowCards before:", yellow_cards_missing_before)
print("Missing YellowCards after:", yellow_cards_missing_after)

print("\nYellowCards data type:")
print(match_statistics_df["YellowCards"].dtype)

print("\nYellowCards value counts:")
print(
    match_statistics_df["YellowCards"]
    .value_counts()
    .sort_index()
)

Missing YellowCards before: 20
Missing YellowCards after: 0

YellowCards data type:
int64

YellowCards value counts:
YellowCards
0    55
1    81
2    41
3    22
4     7
5     2
Name: count, dtype: int64


## 16. Validate Match Statistics Dataset

The MatchStatistics dataset now contains statistics for all 104 matches
and both teams in every match.

Before exporting the dataset, we will validate:

- The dataset contains exactly 208 rows.
- Each MatchID has exactly two teams.
- All MatchIDs exist in Matches.csv.
- All TeamIDs exist in Teams.csv.
- There are no missing values.
- There are no negative statistics.
- ShotsOnTarget does not exceed Shots.
- Possession values are between 0 and 100.

In [ ]:
# Validate the complete MatchStatistics dataset

print("=== MatchStatistics Validation ===\n")

# 1. Row count
print("1. Row count:")
print("Rows:", len(match_statistics_df))
print("Expected:", len(matches_df) * 2)
print(
    "PASS" if len(match_statistics_df) == len(matches_df) * 2
    else "FAIL"
)

# 2. Exactly two rows per MatchID
rows_per_match = match_statistics_df.groupby("MatchID").size()

print("\n2. Teams per match:")
print("Number of MatchIDs:", rows_per_match.nunique())
print(
    "PASS"
    if rows_per_match.eq(2).all()
    else "FAIL"
)

# 3. MatchID validation
invalid_match_ids = set(
    match_statistics_df["MatchID"]
) - set(
    matches_df["MatchID"]
)

print("\n3. Invalid MatchIDs:", len(invalid_match_ids))
print("PASS" if len(invalid_match_ids) == 0 else "FAIL")

# 4. TeamID validation
invalid_team_ids = set(
    match_statistics_df["TeamID"]
) - set(
    teams_df["TeamID"]
)

print("\n4. Invalid TeamIDs:", len(invalid_team_ids))
print("PASS" if len(invalid_team_ids) == 0 else "FAIL")

# 5. Missing values
missing_total = match_statistics_df.isna().sum().sum()

print("\n5. Missing values:", missing_total)
print("PASS" if missing_total == 0 else "FAIL")

# 6. Negative values
numeric_columns = [
    "Possession",
    "Shots",
    "ShotsOnTarget",
    "Corners",
    "Offsides",
    "Fouls",
    "YellowCards",
    "RedCards"
]

negative_values = (
    match_statistics_df[numeric_columns] < 0
).sum().sum()

print("\n6. Negative values:", negative_values)
print("PASS" if negative_values == 0 else "FAIL")

# 7. Shots on target cannot exceed total shots
invalid_shots = (
    match_statistics_df["ShotsOnTarget"]
    > match_statistics_df["Shots"]
).sum()

print("\n7. ShotsOnTarget > Shots:", invalid_shots)
print("PASS" if invalid_shots == 0 else "FAIL")

# 8. Possession must be between 0 and 100
invalid_possession = (
    (match_statistics_df["Possession"] < 0)
    | (match_statistics_df["Possession"] > 100)
).sum()

print("\n8. Invalid possession values:", invalid_possession)
print("PASS" if invalid_possession == 0 else "FAIL")

=== MatchStatistics Validation ===

1. Row count:
Rows: 208
Expected: 208
PASS

2. Teams per match:
Number of MatchIDs: 1
PASS

3. Invalid MatchIDs: 0
PASS

4. Invalid TeamIDs: 0
PASS

5. Missing values: 8
FAIL

6. Negative values: 0
PASS

7. ShotsOnTarget > Shots: 0
PASS

8. Invalid possession values: 0
PASS


### 16.1 — Identify Remaining Missing Values

The validation found 8 missing values in the MatchStatistics dataset.

Before changing any values, we will identify which columns contain the
missing values and inspect the affected records.

No values will be replaced until their meaning has been verified.

In [ ]:
# Identify the remaining missing values

print("Missing values by column:")
print()

missing_by_column = match_statistics_df.isna().sum()

display(
    missing_by_column[
        missing_by_column > 0
    ].to_frame("MissingCount")
)

print("\nTotal missing values:", missing_by_column.sum())

print("\nRows containing missing values:")
display(
    match_statistics_df[
        match_statistics_df.isna().any(axis=1)
    ]
)

Missing values by column:



,MissingCount
Offsides,8



Total missing values: 8

Rows containing missing values:


,MatchID,TeamID,Possession,Shots,ShotsOnTarget,Corners,Offsides,Fouls,YellowCards,RedCards
46,60,32,44,7,2,1,NaN,9,1,0
47,60,13,56,12,5,3,NaN,6,1,0
50,9,17,48,15,4,3,NaN,10,3,0
51,9,28,52,12,1,6,NaN,13,1,0
176,89,32,24,5,1,2,NaN,13,0,0
177,89,6,76,15,5,12,NaN,11,3,0
192,97,6,48,22,8,5,NaN,10,0,0
193,97,21,52,5,1,5,NaN,13,1,0


### 16.2 — Handle Missing Offsides

The remaining 8 missing Offsides values were checked against the source
data and confirmed to represent zero offsides.

Therefore, the missing Offsides values will be converted to 0.

No other statistics will be modified.

In [ ]:
# Convert missing Offsides values to 0

offsides_missing_before = match_statistics_df["Offsides"].isna().sum()

match_statistics_df["Offsides"] = (
    match_statistics_df["Offsides"]
    .fillna(0)
    .astype(int)
)

offsides_missing_after = match_statistics_df["Offsides"].isna().sum()

print("Missing Offsides before:", offsides_missing_before)
print("Missing Offsides after:", offsides_missing_after)

print("\nOffsides data type:")
print(match_statistics_df["Offsides"].dtype)

print("\nOffsides value counts:")
print(
    match_statistics_df["Offsides"]
    .value_counts()
    .sort_index()
)

Missing Offsides before: 8
Missing Offsides after: 0

Offsides data type:
int64

Offsides value counts:
Offsides
0    49
1    59
2    35
3    40
4    13
5     7
6     3
7     1
8     1
Name: count, dtype: int64


## 17. Final MatchStatistics Validation

All previously identified missing values have been reviewed and handled.

We will now perform the final validation of the MatchStatistics dataset
before exporting it.

The dataset must contain 208 rows, valid MatchIDs and TeamIDs, no
missing values, no negative statistics, valid possession values, and
valid ShotsOnTarget values.

In [ ]:
# Final validation of MatchStatistics

print("=== Final MatchStatistics Validation ===\n")

# 1. Row count
expected_rows = len(matches_df) * 2
actual_rows = len(match_statistics_df)

print("1. Row count:")
print("Actual:", actual_rows)
print("Expected:", expected_rows)
print("PASS" if actual_rows == expected_rows else "FAIL")


# 2. Exactly two teams per match
rows_per_match = match_statistics_df.groupby("MatchID").size()

print("\n2. Exactly two rows per MatchID:")
print("Total MatchIDs:", len(rows_per_match))
print("Matches with 2 rows:", rows_per_match.eq(2).sum())
print("PASS" if rows_per_match.eq(2).all() else "FAIL")


# 3. MatchID validation
invalid_match_ids = set(
    match_statistics_df["MatchID"]
) - set(
    matches_df["MatchID"]
)

print("\n3. Invalid MatchIDs:", len(invalid_match_ids))
print("PASS" if len(invalid_match_ids) == 0 else "FAIL")


# 4. TeamID validation
invalid_team_ids = set(
    match_statistics_df["TeamID"]
) - set(
    teams_df["TeamID"]
)

print("\n4. Invalid TeamIDs:", len(invalid_team_ids))
print("PASS" if len(invalid_team_ids) == 0 else "FAIL")


# 5. Missing values
missing_total = match_statistics_df.isna().sum().sum()

print("\n5. Missing values:", missing_total)
print("PASS" if missing_total == 0 else "FAIL")


# 6. Negative values
numeric_columns = [
    "Possession",
    "Shots",
    "ShotsOnTarget",
    "Corners",
    "Offsides",
    "Fouls",
    "YellowCards",
    "RedCards"
]

negative_values = (
    match_statistics_df[numeric_columns] < 0
).sum().sum()

print("\n6. Negative values:", negative_values)
print("PASS" if negative_values == 0 else "FAIL")


# 7. Shots on target cannot exceed total shots
invalid_shots = (
    match_statistics_df["ShotsOnTarget"]
    > match_statistics_df["Shots"]
).sum()

print("\n7. ShotsOnTarget > Shots:", invalid_shots)
print("PASS" if invalid_shots == 0 else "FAIL")


# 8. Possession must be between 0 and 100
invalid_possession = (
    (match_statistics_df["Possession"] < 0)
    | (match_statistics_df["Possession"] > 100)
).sum()

print("\n8. Invalid possession values:", invalid_possession)
print("PASS" if invalid_possession == 0 else "FAIL")


# 9. Duplicate MatchID + TeamID combinations
duplicates = match_statistics_df.duplicated(
    subset=["MatchID", "TeamID"]
).sum()

print("\n9. Duplicate MatchID + TeamID rows:", duplicates)
print("PASS" if duplicates == 0 else "FAIL")

=== Final MatchStatistics Validation ===

1. Row count:
Actual: 208
Expected: 208
PASS

2. Exactly two rows per MatchID:
Total MatchIDs: 104
Matches with 2 rows: 104
PASS

3. Invalid MatchIDs: 0
PASS

4. Invalid TeamIDs: 0
PASS

5. Missing values: 0
PASS

6. Negative values: 0
PASS

7. ShotsOnTarget > Shots: 0
PASS

8. Invalid possession values: 0
PASS

9. Duplicate MatchID + TeamID rows: 0
PASS


## 18. Export MatchStatistics Dataset

The MatchStatistics dataset has passed all validation checks.

We will now export the final dataset to MatchStatistics.csv inside the
project Data folder.

The exported file will be used later for SQL Server, Excel, and Power BI.

In [ ]:
# Export the validated MatchStatistics dataset

match_statistics_file = data_folder / "MatchStatistics.csv"

match_statistics_df.to_csv(
    match_statistics_file,
    index=False
)

print("MatchStatistics.csv exported successfully!")
print("File:", match_statistics_file)
print("Rows:", len(match_statistics_df))
print("Columns:", len(match_statistics_df.columns))

MatchStatistics.csv exported successfully!
File: d:\ai course hw\FIFA_WorldCup_2026_Project\Data\MatchStatistics.csv
Rows: 208
Columns: 10


## 19. Verify Exported MatchStatistics CSV

The validated MatchStatistics dataset has been exported to
MatchStatistics.csv.

We will now reload the exported file and verify its row count,
columns, missing values, and first few records.

In [ ]:
# Reload and verify the exported MatchStatistics.csv

verified_match_statistics_df = pd.read_csv(
    match_statistics_file
)

print("MatchStatistics.csv reloaded successfully!")

print("\nShape:")
print(verified_match_statistics_df.shape)

print("\nColumns:")
print(list(verified_match_statistics_df.columns))

print("\nMissing values:")
print(
    verified_match_statistics_df.isna().sum().sum()
)

print("\nFirst 5 rows:")
display(verified_match_statistics_df.head())

MatchStatistics.csv reloaded successfully!

Shape:
(208, 10)

Columns:
['MatchID', 'TeamID', 'Possession', 'Shots', 'ShotsOnTarget', 'Corners', 'Offsides', 'Fouls', 'YellowCards', 'RedCards']

Missing values:
0

First 5 rows:


,MatchID,TeamID,Possession,Shots,ShotsOnTarget,Corners,Offsides,Fouls,YellowCards,RedCards
0,1,8,61,16,4,3,1,12,1,1
1,1,24,40,3,2,1,1,11,2,2
2,2,31,62,15,6,4,2,9,1,0
3,2,38,39,7,4,5,2,16,0,0
4,25,38,38,14,3,5,2,12,1,0


## 26.8 Inspect MatchStatistics.csv for SQL Import

The SQL Server BULK INSERT provider is unable to read MatchStatistics.csv.

We will inspect the actual exported CSV structure before creating a
SQL-compatible version.

In [15]:
from pathlib import Path
import pandas as pd

data_folder = Path.cwd().parent / "Data"
match_stats_path = data_folder / "MatchStatistics.csv"

print("FILE:", match_stats_path)
print("EXISTS:", match_stats_path.exists())
print("SIZE:", match_stats_path.stat().st_size, "bytes")

# Read the actual CSV
match_stats_check = pd.read_csv(
    match_stats_path,
    encoding="utf-8-sig"
)

print("\nSHAPE:", match_stats_check.shape)

print("\nCOLUMNS:")
print(match_stats_check.columns.tolist())

print("\nDATA TYPES:")
print(match_stats_check.dtypes)

print("\nFIRST 10 ROWS:")
print(match_stats_check.head(10).to_string(index=False))

print("\nMISSING VALUES:")
print(match_stats_check.isna().sum().to_string())

FILE: d:\ai course hw\FIFA_WorldCup_2026_Project\Data\MatchStatistics.csv
EXISTS: True
SIZE: 5324 bytes

SHAPE: (208, 10)

COLUMNS:
['MatchID', 'TeamID', 'Possession', 'Shots', 'ShotsOnTarget', 'Corners', 'Offsides', 'Fouls', 'YellowCards', 'RedCards']

DATA TYPES:
MatchID          int64
TeamID           int64
Possession       int64
Shots            int64
ShotsOnTarget    int64
Corners          int64
Offsides         int64
Fouls            int64
YellowCards      int64
RedCards         int64
dtype: object

FIRST 10 ROWS:
 MatchID  TeamID  Possession  Shots  ShotsOnTarget  Corners  Offsides  Fouls  YellowCards  RedCards
       1       8          61     16              4        3         1     12            1         1
       1      24          40      3              2        1         1     11            2         2
       2      31          62     15              6        4         2      9            1         0
       2      38          39      7              4        5         2     

## 26.9 Create SQL-Compatible MatchStatistics CSV

The original MatchStatistics.csv has been validated successfully and
will remain unchanged.

A separate SQL-compatible copy will be created with:
- UTF-8 encoding
- standard comma-separated format
- no index column
- no quotation marks around ordinary values
- integer values written as integers

This copy will be used only for SQL Server import.

In [16]:
import csv
from pathlib import Path

data_folder = Path.cwd().parent / "Data"

source_path = data_folder / "MatchStatistics.csv"
sql_path = data_folder / "MatchStatistics_SQL.csv"

# Load validated source
df = pd.read_csv(
    source_path,
    encoding="utf-8-sig"
)

# Make sure all values are integers
for column in df.columns:
    df[column] = df[column].astype(int)

# Export SQL-friendly copy
df.to_csv(
    sql_path,
    index=False,
    encoding="utf-8",
    quoting=csv.QUOTE_MINIMAL,
    lineterminator="\n"
)

print("SQL-FRIENDLY CSV CREATED")
print("=" * 100)
print("File:", sql_path)
print("Exists:", sql_path.exists())
print("Shape:", df.shape)
print("Columns:", df.columns.tolist())

SQL-FRIENDLY CSV CREATED
File: d:\ai course hw\FIFA_WorldCup_2026_Project\Data\MatchStatistics_SQL.csv
Exists: True
Shape: (208, 10)
Columns: ['MatchID', 'TeamID', 'Possession', 'Shots', 'ShotsOnTarget', 'Corners', 'Offsides', 'Fouls', 'YellowCards', 'RedCards']


## 26.1 Load MatchStatistics and Add StatisticID

Load the validated MatchStatistics.csv file, add the required
StatisticID primary key, and reorder the columns to match the SQL table.

In [ ]:
from pathlib import Path
import pandas as pd

data_folder = Path.cwd().parent / "Data"
stats_path = data_folder / "MatchStatistics.csv"

# Load the existing MatchStatistics CSV
match_stats_sql_df = pd.read_csv(stats_path)

# Add StatisticID as the SQL primary key
match_stats_sql_df.insert(
    0,
    "StatisticID",
    range(1, len(match_stats_sql_df) + 1)
)

# Reorder columns to exactly match the SQL table
match_stats_sql_df = match_stats_sql_df[
    [
        "StatisticID",
        "MatchID",
        "TeamID",
        "Shots",
        "ShotsOnTarget",
        "Possession",
        "Corners",
        "Fouls",
        "YellowCards",
        "RedCards",
        "Offsides"
    ]
]

print("MATCH STATISTICS SQL DATASET")
print("=" * 100)

print("Shape:", match_stats_sql_df.shape)

print("\nColumns:")
print(match_stats_sql_df.columns.tolist())

print("\nFirst 10 rows:")
print(match_stats_sql_df.head(10).to_string(index=False))

print("\nMissing values:")
print(match_stats_sql_df.isna().sum().to_string())

print("\nStatisticID duplicates:",
      match_stats_sql_df["StatisticID"].duplicated().sum())

print(
    "StatisticID range:",
    match_stats_sql_df["StatisticID"].min(),
    "to",
    match_stats_sql_df["StatisticID"].max()
)

## 26.2 Export Corrected MatchStatistics CSV

Overwrite the existing MatchStatistics.csv with the SQL-ready version,
including StatisticID and the SQL table column order.

In [ ]:
from pathlib import Path

data_folder = Path.cwd().parent / "Data"
stats_path = data_folder / "MatchStatistics.csv"

# Overwrite the existing CSV
match_stats_sql_df.to_csv(
    stats_path,
    index=False,
    encoding="utf-8-sig",
    lineterminator="\r\n"
)

print("MATCH STATISTICS CSV OVERWRITTEN")
print("=" * 100)
print("File:", stats_path)
print("Exists:", stats_path.exists())
print("Size:", stats_path.stat().st_size, "bytes")
print("Rows:", len(match_stats_sql_df))
print("Columns:", len(match_stats_sql_df.columns))
print("Columns:", match_stats_sql_df.columns.tolist())

## 26.3 Verify Exported MatchStatistics CSV

Reload the exported CSV and verify that the saved file has the correct
columns, row count, primary key, and no missing values.

In [ ]:
import pandas as pd
from pathlib import Path

data_folder = Path.cwd().parent / "Data"
stats_path = data_folder / "MatchStatistics.csv"

# Reload the actual saved CSV
match_stats_verify = pd.read_csv(stats_path)

print("MATCH STATISTICS CSV VERIFICATION")
print("=" * 100)

print("Shape:", match_stats_verify.shape)

print("\nColumns:")
print(match_stats_verify.columns.tolist())

print("\nMissing values:")
print(match_stats_verify.isna().sum().to_string())

print("\nDuplicate StatisticIDs:",
      match_stats_verify["StatisticID"].duplicated().sum())

print(
    "\nStatisticID range:",
    match_stats_verify["StatisticID"].min(),
    "to",
    match_stats_verify["StatisticID"].max()
)

print("\nFirst 5 rows:")
print(match_stats_verify.head().to_string(index=False))

print("\nVerification:")
print("208 rows:", len(match_stats_verify) == 208)
print("11 columns:", len(match_stats_verify.columns) == 11)
print("No missing values:", match_stats_verify.isna().sum().sum() == 0)
print("StatisticID unique:",
      match_stats_verify["StatisticID"].is_unique)